In [0]:
-- Optimized fact_orders table with only KPI-relevant columns
CREATE OR REPLACE TABLE automobile_catalog.003_gold.fact_orders AS
SELECT
    o.store_id,
    o.technician_id,
    o.service_type,
    o.order_status,
    
    -- Calculated metrics used in KPIs
    CASE 
        WHEN o.vehicle_in_datetime IS NOT NULL 
         AND o.vehicle_out_datetime IS NOT NULL
        THEN DATEDIFF(o.vehicle_out_datetime, o.vehicle_in_datetime)
        ELSE NULL
    END AS days_in_shop,

    CASE 
        WHEN o.vehicle_in_datetime IS NOT NULL 
         AND o.actual_work_start_datetime IS NOT NULL
        THEN DATEDIFF(o.actual_work_start_datetime, o.vehicle_in_datetime)
        ELSE NULL
    END AS days_to_work_start,

    CASE 
        WHEN o.actual_work_start_datetime IS NOT NULL 
         AND o.actual_completion_datetime IS NOT NULL
        THEN DATEDIFF(o.actual_completion_datetime, o.actual_work_start_datetime)
        ELSE NULL
    END AS work_duration_days,

    CASE 
        WHEN o.promised_delivery_datetime IS NOT NULL
         AND o.actual_delivery_datetime IS NOT NULL
        THEN DATEDIFF(o.actual_delivery_datetime, o.promised_delivery_datetime)
        ELSE NULL
    END AS delivery_variance_days,

    CASE 
        WHEN o.actual_delivery_datetime <= o.promised_delivery_datetime
        THEN TRUE
        ELSE FALSE
    END AS is_on_time,

    -- Date for time-based aggregations
    o.vehicle_in_datetime

FROM automobile_catalog.002_silver.order AS o;

SELECT COUNT(*) as total_rows, COUNT(DISTINCT store_id) as unique_stores, COUNT(DISTINCT technician_id) as unique_technicians FROM automobile_catalog.003_gold.fact_orders;

In [0]:
-- Optimized fact_invoices table with only KPI-relevant columns
CREATE OR REPLACE TABLE automobile_catalog.003_gold.fact_invoices AS
SELECT
    i.order_id,
    o.store_id,
    o.order_status,
    i.invoice_date,
    i.invoice_amount

FROM automobile_catalog.002_silver.invoice AS i
INNER JOIN automobile_catalog.002_silver.order AS o
    ON i.order_id = o.order_id;

SELECT * FROM automobile_catalog.003_gold.fact_invoices LIMIT 5;

In [0]:
-- Optimized fact_estimates table with only KPI-relevant columns
CREATE OR REPLACE TABLE automobile_catalog.003_gold.fact_estimates AS
WITH invoice_amt AS (
    SELECT 
        order_id,
        invoice_amount AS actual_amount
    FROM automobile_catalog.002_silver.invoice
)

SELECT
    e.order_id,
    o.store_id,
    e.estimator_id,
    CAST(e.created_at AS DATE) AS estimate_date,
    
    -- Initial Estimate Flag
    CASE 
        WHEN e.version_no = 1 THEN TRUE
        ELSE FALSE
    END AS is_initial_estimate,
    
    -- Amounts (variance calculated in cubes)
    e.estimate_amount,
    ia.actual_amount

FROM automobile_catalog.002_silver.estimate AS e
INNER JOIN automobile_catalog.002_silver.order AS o
    ON e.order_id = o.order_id
LEFT JOIN invoice_amt AS ia
    ON e.order_id = ia.order_id;

SELECT * FROM automobile_catalog.003_gold.fact_estimates LIMIT 5;

In [0]:
-- Optimized fact_survey_responses table with only KPI-relevant columns
CREATE OR REPLACE TABLE automobile_catalog.003_gold.fact_survey_responses AS
SELECT
    o.store_id,
    cs.survey_sent_date,
    cs.responded_flag,
    
    -- Individual Ratings (used in satisfaction KPIs)
    cs.delivered_on_time_rating,
    cs.work_quality_rating,
    cs.cleanliness_rating,
    cs.communication_rating,
    cs.overall_satisfaction_rating

FROM automobile_catalog.002_silver.customer_survey AS cs
INNER JOIN automobile_catalog.002_silver.order AS o
    ON cs.order_id = o.order_id;

SELECT COUNT(*) as total_rows, COUNT(DISTINCT store_id) as unique_stores FROM automobile_catalog.003_gold.fact_survey_responses;

In [0]:
-- Optimized fact_budget table with only KPI-relevant columns
CREATE OR REPLACE TABLE automobile_catalog.003_gold.fact_budget AS
SELECT
    ns_store_id AS store_id,
    DATE_FORMAT(month, 'yyyy-MM') AS budget_month,
    budget_amount

FROM automobile_catalog.002_silver.ns_budget;

SELECT * FROM automobile_catalog.003_gold.fact_budget LIMIT 5;